In [15]:
import pandas as pd
import numpy as np
# Load dataset
df = pd.read_csv(r"C:\\college_recommendation_system\\CleanData\\Cleaned_tgeapcet_2017_2024.csv")

df["category_gender"] = (
    df["category"].astype(str) + "_" + df["gender"].astype(str)
)

print(df.columns)
df.isnull().sum()
# Drop college_type and year_of_estab columns
df = df.drop(columns=["college_type", "year_of_estab"])
df.isnull().sum()
# Remove extra spaces
df["co_education"] = df["co_education"].str.strip()
df["category"] = df["category"].str.strip()

# Convert to uppercase
df["co_education"] = df["co_education"].str.upper()
df["category"] = df["category"].str.upper()

from sklearn.preprocessing import LabelEncoder

le_college = LabelEncoder()
df["inst_code"] = le_college.fit_transform(df["inst_code"])


le_branch = LabelEncoder()
df["branch_code"] = le_branch.fit_transform(df["branch_code"])

df["category"] = df["category"].fillna("EWS")

category_map = {
    "OC":0,
    "BC_A":1,
    "BC_B":2,
    "BC_C":3,
    "BC_D":4,
    "BC_E":5,
    "SC":6,
    "ST":7,
    "EWS":8
}

df["category"] = df["category"].map(category_map)

gender_map = {
    "COED":0,
    "GIRLS":1
}
df["co_education"] = df["co_education"].map(gender_map)

le_location = LabelEncoder()
df["dist_code"] = le_location.fit_transform(df["dist_code"])

gender_map = {
    "Boys": 0,
    "Girls": 1
}

df["gender"] = df["gender"].map(gender_map)

df = df.drop(columns=[
    "institute_name",
    "branch_name",
    "category",
    "gender",
    "place"
])


df["category_gender"] = df["category_gender"].str.strip().str.upper()

from sklearn.preprocessing import LabelEncoder

le_catgen = LabelEncoder()
df["catgen_id"] = le_catgen.fit_transform(df["category_gender"])

df = df.drop(columns=["category_gender"])

df.head()

Index(['inst_code', 'institute_name', 'branch_code', 'branch_name',
       'co_education', 'college_type', 'year_of_estab', 'place', 'dist_code',
       'year', 'cutoff', 'category', 'gender', 'category_gender'],
      dtype='object')


,inst_code,branch_code,co_education,dist_code,year,cutoff,catgen_id
0,0,20,0,2,2017,26907.0,12
1,0,30,0,2,2017,33871.0,12
2,0,33,0,2,2017,39938.0,12
3,0,46,0,2,2017,38774.0,12
4,1,13,0,11,2017,33769.0,12


In [17]:
# ==============================
# 1️⃣ Import Libraries
# ==============================
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score

# ==============================
# 2️⃣ Sort Properly
# ==============================
df = df.sort_values(
    ["inst_code", "branch_code", "catgen_id", "year"]
).reset_index(drop=True)

# ==============================
# 3️⃣ Feature Engineering
# ==============================

# Previous year cutoff
df["prev_cutoff"] = (
    df.groupby(["inst_code", "branch_code", "catgen_id"])["cutoff"]
    .shift(1)
)

# 2-year lag
df["prev_cutoff_2"] = (
    df.groupby(["inst_code", "branch_code", "catgen_id"])["cutoff"]
    .shift(2)
)

df["rolling_mean_3"] = (
    df.groupby(["inst_code", "branch_code", "catgen_id"])["cutoff"]
    .shift(1)
    .rolling(3)
    .mean()
)

# Drop NaNs (from lag)
df = df.dropna().reset_index(drop=True)

# ==============================
# 4️⃣ Define Features
# ==============================
features = [
    "inst_code",
    "branch_code",
    "co_education",
    "dist_code",
    "year",
    "catgen_id",
    "prev_cutoff",
    "prev_cutoff_2",
    "rolling_mean_3"
]

# ==============================
# 5️⃣ Time-Based Split
# ==============================
train = df[df["year"] < 2024]
test = df[df["year"] == 2024]

X_train = train[features]
y_train = train["cutoff"]

X_test = test[features]
y_test = test["cutoff"]

# ==============================
# 6️⃣ Scale Features (Important for LR)
# ==============================
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ==============================
# 7️⃣ Train Linear Regression
# ==============================
model = LinearRegression()
model.fit(X_train_scaled, y_train)

# ==============================
# 8️⃣ Predict & Evaluate
# ==============================
y_pred = model.predict(X_test_scaled)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Linear Regression MAE:", mae)
print("Linear Regression R2 :", r2)

Linear Regression MAE: 26951.46947743314
Linear Regression R2 : 0.5460258607470414
